# Finsheild Phase 1 — EDA (local)
Runs locally with synthetic or real data. Generates figures under evaluation/figures and stats for docs.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from finsheild.data.loader import load_raw
from finsheild.data.splits import make_splits
from finsheild.data.preprocessing import FraudPreprocessor

df = load_raw("data/raw/creditcard.csv")
print(f"Shape: {df.shape}")
print(df.dtypes)
print(f"Missing: {df.isnull().sum().sum()}")
print(f"Duplicated: {df.duplicated().sum()}")
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True))


In [ ]:
# Class distribution bar
Path("evaluation/figures").mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(4,3))
ax = sns.countplot(x="Class", data=df)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2., p.get_height()), ha='center', va='bottom')
plt.title(f"Class dist — fraud {df['Class'].mean():.2%}")
plt.savefig("evaluation/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Amount/Time by class
fig, axes = plt.subplots(2,2, figsize=(12,8))
for col, ax in zip(["Amount","Time"], axes[0]):
    df.boxplot(column=col, by="Class", ax=ax)
    ax.set_title(col)
for col, ax in zip(["Amount","Time"], axes[1]):
    for cls in [0,1]:
        sns.histplot(df.loc[df["Class"]==cls, col], bins=50, kde=False, ax=ax, label=f"Class {cls}", alpha=0.5)
    ax.set_title(f"{col} hist by Class")
    ax.legend()
plt.suptitle("")
plt.tight_layout()
plt.savefig("evaluation/figures/amount_time_by_class.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Correlation / PCA note for V1-V28
corr = df[[f"V{i}" for i in range(1,29)]+["Amount","Time"]].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-0.5, vmax=0.5)
plt.title("Correlation — V1-V28 + Amount/Time")
plt.tight_layout()
plt.savefig("evaluation/figures/correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("V1-V28 are PCA-transformed (anonymized); low pairwise corr confirms decorrelation. See docs/dataset.md.")


In [ ]:
# Splits + preprocessing demo
from finsheild.data.splits import save_splits
train, val, test = make_splits(df, test_size=0.15, val_size=0.15, random_state=42)
print(train.shape, val.shape, test.shape)
pre = FraudPreprocessor()
train_t = pre.fit_transform_train(train)
val_t = pre.transform(val)
test_t = pre.transform(test)
print("Means after scaling (train):", train_t[["Amount","Time"]].mean().to_dict())
save_splits(train_t, val_t, test_t, out_dir="data/processed")
pre.save("data/processed/scaler.joblib")
